In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.neighbors import BallTree

In [2]:
df = pd.read_csv("bike_trip_data(July 2025-June 2026).csv")

In [3]:
df = df.drop(columns=['Unnamed: 0'])

In [4]:
data_summary = []
data_missing = []
space_str = []
data_coor_outlier = []
ins_archive = []
ins_0 = []

df_sum = pd.DataFrame(data_summary, columns=['Condition','Value'])
df_miss = pd.DataFrame(data_missing, columns=['Variables','Total_Missing'])
df_space = pd.DataFrame(space_str, columns=['Variables', 'Value with Extra Space'])
df_coor_outlier = pd.DataFrame(space_str, columns=['Variables', 'Outlier'])
df_arch = pd.DataFrame(ins_archive, columns=['variables', 'val_w_archive'])
df_0 = pd.DataFrame(ins_0, columns=['variables', 'val_w_0'])
df_dtype = pd.DataFrame(df.dtypes, columns=['type']).reset_index()
df_dtype.rename(columns={'index':'Variables'}, inplace=True)

df_sum.loc[len(df_sum)] = ['Total Records', df.shape[0]]
df_sum.loc[len(df_sum)] = [f'Duplicate Data', df.duplicated().sum()]
df_sum.loc[len(df_sum)] = [f'Need to Swap ("started_at" & "ended_at Value")', (df['started_at'] > df['ended_at']).sum()]

for miss in df.isna():
    df_miss.loc[len(df_miss)] = [miss, df[miss].isna().sum()]

for col in df.columns:
    has_extra_space = df[col].astype(str).str.contains(r'^\s+|\s+$|\s{2,}', regex=True)
    df_bermasalah = df[has_extra_space] # Menampilkan baris yang memiliki spasi ekstra
    df_space.loc[len(df_space)] = [col,len(df_bermasalah)]

for col in df.columns[8:12]:
    if col[-3:] == 'lat':
        outlier_lat = (round(df[col], 2) <= 41.60) | (round(df[col], 2) >= 42.10)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lat.sum()]
    else:
        outlier_lng = (round(df[col], 2) <= -88.00) | (round(df[col], 2) >= -87.50)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lng.sum()]

for col in df.columns[4:8]:
    archived = df[col].str.contains('Archive', case = False, na=False).sum()
    df_arch.loc[len(df_arch)] = [col,archived]

for col in df.columns[4:8]:
    archived = df[col].str.startswith('0', na=False).sum()
    df_0.loc[len(df_0)] = [col,archived]

In [5]:
html_str_1 = f"""
<h1> Dataset Check </h1>
<div style="display: grid; gap: 20px;">
    <div style="display: flex; gap: 20px;">
        <div><h3>Summary</h3>{df_sum.to_html()}</div>
        <div><h3>Coordinate Outlier</h3>{df_coor_outlier.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Missing Value</h3>{df_miss.to_html()}</div>
        <div><h3>Extra Spaces</h3>{df_space.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Value with "Archive"</h3>{df_arch.to_html()}</div>
        <div><h3>Value start with "0"</h3>{df_0.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Data Type</h3>{df_dtype.to_html()}</div>
    </div>
</div>
"""

display(HTML(html_str_1))

,Condition,Value
0,Total Records,5932350
1,Duplicate Data,35
2,"Need to Swap (""started_at"" & ""ended_at Value"")",29
,Variables,Outlier
0,start_lat,0
1,start_lng,0
2,end_lat,18
3,end_lng,8
,Variables,Total_Missing
0,ride_id,1


#### Cleaning Tasks
1. **Remove Duplicates**
    - Remove 35 duplicate records from the dataset

2. **Fix Time Recording Errors**
    - Swap `started_at` and `ended_at` values for 29 records where start time is after end time

3. **Remove or Handle Missing Values**
    - Remove records with missing end station coordinates (`end_lat` and `end_lng`) - 5,607 records (6 records already remove because its duplicated)
    - Decide on handling missing station names and IDs (1.2M+ records)

4. **Text Cleaning and Standardization**
    - Strip leading and trailing spaces from station name
    - Remove "[Archive]" tags and associated random numbers from station values
    - Convert all station names to lowercase for consistency

5. **Remove Coordinate Outliers**
    - Remove 18 records with invalid ending latitude values
    - Remove 8 records with invalid ending longitude values

6. **Rename Columns**
    - Rename `ride_id` to `trip_id`
    - Rename `rideable_type` to `bike_type`

7. **Convert Data Types**
    - Convert `started_at` and `ended_at` to datetime format
    - Convert categorical columns (`bike_type`, `start_station_name`, `end_station_name`, `member_casual`) to category data type
8. **Remove Columns**
    - Remove `start_station_id` and `end_station_id` columns

    
#### Expected Outcome
A clean, standardized dataset ready for exploratory data analysis and visualization.

In [6]:
df_cleaned = df

In [7]:
df_cleaned['start_station_name'] = df['start_station_name'].str.lower()
df_cleaned['end_station_name'] = df['end_station_name'].str.lower()
df_cleaned = df.drop_duplicates()

In [8]:
condition = df_cleaned['started_at'] > df_cleaned['ended_at']
df_cleaned.loc[condition, ['started_at', 'ended_at']] = df_cleaned.loc[condition, ['ended_at', 'started_at']].values

In [9]:
condition = df_cleaned[['end_station_name', 'end_lat']].isna().all(axis=1)
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [10]:
station_ref = df_cleaned[
    df_cleaned['start_station_name'].notna() & 
    df_cleaned['start_station_id'].notna()
].drop_duplicates(subset=['start_lat', 'start_lng'])[
    ['start_lat', 'start_lng', 'start_station_name', 'start_station_id']
]

station_ref = station_ref.drop_duplicates(subset=['start_station_id'])
station_ref = station_ref.drop_duplicates(subset=['start_station_name'])
station_ref = station_ref.reset_index()
station_ref = station_ref.drop(columns=['index'])

In [11]:
#MACHINE LEARNING
mask_null = df_cleaned['start_station_name'].isna()
df_null = df_cleaned[mask_null].copy()

latlon_ref = np.radians(station_ref[['start_lat', 'start_lng']].values)
latlon_null = np.radians(df_null[['start_lat', 'start_lng']].values)

tree = BallTree(latlon_ref, metric='haversine')

In [12]:
distances, indices = tree.query(latlon_null, k=2)

In [13]:
distances_km = distances * 6371.0

max_distances_km = 1.8
margin_km = 0.005

In [14]:
station_result = []

for i in range(len(df_null)):
    dist1, dist2 = distances_km[i][0], distances_km[i][1]
    idx1, idx2 = indices[i][0], indices[i][1]

    if dist1 > max_distances_km:
        station_result.append(np.nan)

    elif (dist2-dist1) <= margin_km:
        name1 = station_ref.iloc[idx1]['start_station_name']
        name2 = station_ref.iloc[idx2]['start_station_name']
        station_result.append(f'Ambiguous: {name1} / {name2}')

    else:
        station_result.append(station_ref.iloc[idx1]['start_station_name'])

df_cleaned.loc[mask_null, 'start_station_name'] = station_result

In [15]:
condition = df_cleaned['start_station_name'].isna()
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [16]:
df_cleaned = df_cleaned.drop(columns=['start_station_id', 'end_station_id'])

In [17]:
station_ref = df_cleaned[
    df_cleaned['end_station_name'].notna()
].drop_duplicates(subset=['end_lat', 'end_lng'])[
    ['end_lat', 'end_lng', 'end_station_name']
]

station_ref = station_ref.drop_duplicates(subset=['end_station_name'])
station_ref = station_ref.reset_index()
station_ref = station_ref.drop(columns=['index'])

In [18]:
#MACHINE LEARNING
mask_null = df_cleaned['end_station_name'].isna()
df_null = df_cleaned[mask_null].copy()

latlon_ref = np.radians(station_ref[['end_lat', 'end_lng']].values)
latlon_null = np.radians(df_null[['end_lat', 'end_lng']].values)

tree = BallTree(latlon_ref, metric='haversine')

In [19]:
distances, indices = tree.query(latlon_null, k=2)

In [20]:
distances_km = distances * 6371.0

max_distances_km = 1.8
margin_km = 0.005

In [21]:
station_result = []

for i in range(len(df_null)):
    dist1, dist2 = distances_km[i][0], distances_km[i][1]
    idx1, idx2 = indices[i][0], indices[i][1]

    if dist1 > max_distances_km:
        station_result.append(np.nan)

    elif (dist2-dist1) <= margin_km:
        name1 = station_ref.iloc[idx1]['end_station_name']
        name2 = station_ref.iloc[idx2]['end_station_name']
        station_result.append(f'Ambiguous: {name1} / {name2}')

    else:
        station_result.append(station_ref.iloc[idx1]['end_station_name'])

df_cleaned.loc[mask_null, 'end_station_name'] = station_result

In [22]:
condition = df_cleaned['end_station_name'].isna()
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [23]:
for col in df_cleaned.columns[4:6]:
    print(col)

start_station_name
end_station_name


In [24]:
for col in df_cleaned.columns[4:6]:
    df_cleaned[col] = df_cleaned[col].str.strip()
    df_cleaned[col] = df_cleaned[col].str.replace(r'^\s+|\s+$|\s{2,}', ' ', regex=True)

In [25]:
#outlier sudah terhapus saat proses yang lain

In [26]:
#rename columns
df_cleaned.rename(columns={'ride_id': 'trip_id', 'rideable_type': 'bike_type'}, inplace=True)

In [27]:
#convert data type
df_cleaned = df_cleaned.astype({
    'trip_id': 'category',
    'bike_type': 'category',
    'start_station_name': 'category',
    'end_station_name': 'category',
    })

In [28]:
df_cleaned['started_at'] = pd.to_datetime(df_cleaned['started_at'])
df_cleaned['ended_at'] = pd.to_datetime(df_cleaned['ended_at'])

In [29]:
# Save the cleaned trip data to a CSV file in the current working directory.
df_cleaned.to_csv("bike_trip_data_clean(July 2025-June 2026).csv", index=False)